# Small-model Agentic RAG on Kaggle: failure analysis, small-model benchmark, tool-calling QLoRA

**One pipeline, three contributions**

1. **Agentic RAG baseline** - no retrieval vs fixed RAG vs an agent that decides when and what to search (multi-hop QA).
2. **Small-model benchmark** - the same agent driven by 4-bit Qwen2.5 3B / 7B (optionally 14B); automatic failure taxonomy per model size.
3. **Tool-calling QLoRA** - the 3B student is fine-tuned on filtered traces produced by a larger teacher, then tested on HotpotQA dev and on two **out-of-domain** sets (MuSiQue, 2WikiMultihopQA).

## How to run (read once)

1. Right panel -> **Session options**: Accelerator = **GPU T4 x2** (or P100), **Internet = On** (needs a phone-verified account).
2. **First run with `SMOKE_TEST = True`** (default). It uses ~24 questions and the 3B model only, and finishes in about 20-40 minutes. It writes to a separate `smoke/` folder, so it never pollutes the real run.
3. Set `SMOKE_TEST = False`, then **Save Version -> Save & Run All (Commit)**. The commit run continues in the background even if you close the browser.
4. Everything is **resumable**: every finished condition is written to `/kaggle/working/outputs/<run>/traces/`. If the session or the weekly quota ends, start a new session with the previous version's output attached as input (or just re-run on the same version) and the finished parts are skipped.
5. Download `outputs_<run>_bundle.zip` (made by the last cell) and read `RESULTS.md` first.

**Stages** (edit `STAGES` in the config cell): `tune` (optional prompt/top-k sweep on held-out questions), `base_eval`, `traces`, `finetune`, `ft_eval`, `ablate` (optional), `report`.

*Honest limitations:* the corpus is the pooled HotpotQA *distractor* paragraphs of the evaluation questions (not the 5M-paragraph full Wikipedia), so absolute scores are higher than open-domain numbers. Compare **conditions and model sizes against each other**, not against leaderboard numbers.

In [1]:
# ---- 1. Install (run once per fresh session; set INSTALL=False after that) ----
import subprocess, sys
INSTALL = True
if INSTALL:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                    "bitsandbytes", "peft", "accelerate", "sentence-transformers", "datasets"], check=False)
    print("Install done. If you later see an ImportError, use Run -> Restart session, set INSTALL=False and run again.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
gradio 6.26.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


Install done. If you later see an ImportError, use Run -> Restart session, set INSTALL=False and run again.


In [2]:
# ---- 2. Configuration (the only cell you normally edit) ----
import os, sys, json, re, string, math, time, random, gc, shutil, hashlib, glob, collections
from collections import Counter
from pathlib import Path
import numpy as np

CFG = dict(
    SMOKE_TEST   = True,      # True = tiny end-to-end test. Set False for the real run.
    RUN_TAG      = "run1",    # output sub-folder; change it to keep several runs side by side
    SEED         = 42,
    FORCE        = False,     # True = recompute even if a trace file already exists

    # stages: pick from tune, base_eval, traces, finetune, ft_eval, ablate, report
    STAGES       = ["base_eval", "traces", "finetune", "ft_eval", "report"],

    # models (4-bit NF4 via bitsandbytes). Swap in any chat model of the same family.
    MODELS       = {"3B": "Qwen/Qwen2.5-3B-Instruct",
                    "7B": "Qwen/Qwen2.5-7B-Instruct",
                    "14B": "Qwen/Qwen2.5-14B-Instruct"},   # 14B is slow; add "14B" to EVAL_MODELS only if you have quota
    EVAL_MODELS  = ["3B", "7B"],
    TEACHER      = "7B",      # generates the fine-tuning traces (must be in EVAL_MODELS)
    FT_BASE      = "3B",      # student that gets LoRA-tuned (must be in EVAL_MODELS)
    BATCH        = {"3B": 16, "7B": 8, "14B": 4},          # generation batch size (auto-halves on OOM)

    # retrieval / agent
    EMB_MODEL    = "BAAI/bge-small-en-v1.5",
    TOP_K        = 3,         # passages returned per search call
    FIXED_RAG_K  = 5,         # passages for the fixed-RAG baseline
    MAX_STEPS    = 6,         # assistant turns per question (<= 5 searches + finish)
    MAX_NEW_TOK  = 96,
    PASSAGE_WORDS= 90,        # passage truncation inside observations
    PROMPT       = "base",    # "base" or "decompose" (the tune stage can pick for you)

    # data sizes
    N_EVAL       = 500,       # HotpotQA dev questions used for the final numbers
    N_TUNE       = 150,       # separate dev questions, only for the tune stage
    N_TRAIN_Q    = 2000,      # HotpotQA train questions the teacher solves
    N_OOD        = 300,       # questions per out-of-domain set
    TUNE_MODEL   = "3B",

    # QLoRA
    LORA_R=16, LORA_ALPHA=32, LORA_DROPOUT=0.05, LR=2e-4, EPOCHS=2,
    GRAD_ACC=16, MAX_SEQ_LEN=3072, MAX_TRAIN_TRACES=3000,
    REQUIRE_GOLD_RECALL = True,   # SFT filter: keep a trace only if its searches actually retrieved all gold paragraphs
    ABL_SIZES    = [250, 500],  # training-set sizes for the ablate stage

    SESSION_BUDGET_H = 8.5,   # stop cleanly before the Kaggle session limit (9-12 h)
)

if CFG["SMOKE_TEST"]:
    CFG.update(N_EVAL=24, N_TUNE=16, N_TRAIN_Q=60, N_OOD=12, EPOCHS=1, EVAL_MODELS=["3B"],
               TEACHER="3B", FT_BASE="3B", MAX_TRAIN_TRACES=40, ABL_SIZES=[10], GRAD_ACC=4)

SEED = CFG["SEED"]
STAGES = CFG["STAGES"]
IN_KAGGLE = os.path.exists("/kaggle")
BASE_OUT = "/kaggle/working" if IN_KAGGLE else "./working"
RUN_NAME = "smoke" if CFG["SMOKE_TEST"] else CFG["RUN_TAG"]
OUT = os.path.join(BASE_OUT, "outputs", RUN_NAME)
for sub in ["traces", "sft", "adapters", "figs", "emb", "logs"]:
    os.makedirs(os.path.join(OUT, sub), exist_ok=True)

T0 = time.time()
def time_left_h():
    return CFG["SESSION_BUDGET_H"] - (time.time() - T0) / 3600.0

class BudgetExceeded(Exception):
    pass

def check_budget(min_h=0.25):
    if time_left_h() < min_h:
        raise BudgetExceeded("Session budget almost used up. Everything finished so far is saved; "
                             "start a new session and re-run - finished parts are skipped.")

random.seed(SEED); np.random.seed(SEED)
print("run:", RUN_NAME, "| output:", OUT)
print("stages:", STAGES)

run: smoke | output: /kaggle/working/outputs/smoke
stages: ['base_eval', 'traces', 'finetune', 'ft_eval', 'report']


In [3]:
# ---- 3. GPU check, optional HF login ----
import torch
assert torch.cuda.is_available(), "No GPU. Session options -> Accelerator -> GPU T4 x2 (or P100)."
torch.manual_seed(SEED)
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name}  {p.total_memory/2**30:.1f} GiB  compute capability {p.major}.{p.minor}")
print("torch", torch.__version__)
try:  # optional: a read token avoids anonymous rate limits. Add a Kaggle secret named HF_TOKEN.
    from kaggle_secrets import UserSecretsClient
    tok_ = UserSecretsClient().get_secret("HF_TOKEN")
    from huggingface_hub import login
    login(tok_)
    print("Logged in to Hugging Face.")
except Exception:
    print("No HF_TOKEN secret found (fine - all models/datasets used here are public).")

GPU 0: Tesla T4  14.6 GiB  compute capability 7.5
GPU 1: Tesla T4  14.6 GiB  compute capability 7.5
torch 2.11.0+cu128
No HF_TOKEN secret found (fine - all models/datasets used here are public).


In [4]:
# ---- 4. Utilities: answer metrics, jsonl io, bootstrap ----
def save_jsonl(path, rows):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

def normalize_answer(s):
    s = (s or "").lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())

def f1_single(pred, gold):
    p, g = normalize_answer(pred), normalize_answer(gold)
    if p in ("yes", "no", "noanswer") and p != g:
        return 0.0
    if g in ("yes", "no", "noanswer") and p != g:
        return 0.0
    pt, gt = p.split(), g.split()
    common = Counter(pt) & Counter(gt)
    ns = sum(common.values())
    if ns == 0:
        return 0.0
    prec, rec = ns / len(pt), ns / len(gt)
    return 2 * prec * rec / (prec + rec)

def score_answer(pred, answers):
    # returns (EM, F1, Acc); Acc = normalised gold is contained in the prediction (lenient)
    if pred is None:
        return 0.0, 0.0, 0.0
    em = max(float(normalize_answer(pred) == normalize_answer(a)) for a in answers)
    f1 = max(f1_single(pred, a) for a in answers)
    acc = max(float(bool(normalize_answer(a)) and (normalize_answer(a) in normalize_answer(pred))) for a in answers)
    return em, f1, acc

def bootstrap_ci(x, iters=2000, seed=0):
    x = np.asarray(x, dtype=float)
    if len(x) == 0:
        return (float("nan"), float("nan"))
    rng = np.random.default_rng(seed)
    means = x[rng.integers(0, len(x), (iters, len(x)))].mean(axis=1)
    return tuple(np.percentile(means, [2.5, 97.5]))

def paired_bootstrap(a, b, iters=4000, seed=0):
    # a, b: per-question scores of system A and B on the SAME questions. Returns mean(b-a), 95% CI, P(diff<=0)
    d = np.asarray(b, dtype=float) - np.asarray(a, dtype=float)
    rng = np.random.default_rng(seed)
    means = d[rng.integers(0, len(d), (iters, len(d)))].mean(axis=1)
    return float(d.mean()), tuple(np.percentile(means, [2.5, 97.5])), float((means <= 0).mean())

In [5]:
# ---- 5. Datasets -> one unified format ----
# unified example: {id, question, answers[list], paragraphs[(title, text)], gold_titles[list]}
def conv_hotpot(ex):
    ctx = ex["context"]
    paras = [(t, " ".join(s).strip()) for t, s in zip(ctx["title"], ctx["sentences"])]
    gold = sorted(set(ex["supporting_facts"]["title"]))
    return dict(id=str(ex["id"]), question=ex["question"], answers=[ex["answer"]], paragraphs=paras, gold_titles=gold)

def conv_musique(ex):
    paras = [(p["title"], p["paragraph_text"].strip()) for p in ex["paragraphs"]]
    gold = sorted(set(p["title"] for p in ex["paragraphs"] if p.get("is_supporting")))
    answers = [ex["answer"]] + [a for a in (ex.get("answer_aliases") or []) if a]
    return dict(id=str(ex["id"]), question=ex["question"], answers=answers, paragraphs=paras, gold_titles=gold)

def conv_2wiki(ex):
    ctx = ex["context"]
    if isinstance(ctx, dict):
        titles = ctx.get("title", [])
        contents = ctx.get("content", ctx.get("sentences", []))
        pairs = list(zip(titles, contents))
    else:
        pairs = [(c[0], c[1]) for c in ctx]
    paras = [(t, (" ".join(c) if isinstance(c, (list, tuple)) else str(c)).strip()) for t, c in pairs]
    sf = ex.get("supporting_facts")
    if isinstance(sf, dict):
        gold = sorted(set(sf.get("title", [])))
    elif sf:
        gold = sorted(set(x[0] for x in sf))
    else:
        gold = []
    return dict(id=str(ex.get("_id", ex.get("id", ""))), question=ex["question"], answers=[ex["answer"]],
                paragraphs=paras, gold_titles=gold)

def load_hotpot_split(split):
    from datasets import load_dataset
    errs = []
    for kw in ({}, {"revision": "refs/convert/parquet"}):
        try:
            return load_dataset("hotpotqa/hotpot_qa", "distractor", split=split, **kw)
        except Exception as e:
            errs.append(repr(e)[:300])
    raise RuntimeError("Could not load HotpotQA: " + " | ".join(errs) +
                       "\nCheck that Internet is ON in Session options, or attach a HotpotQA copy as a Kaggle dataset.")

def load_cb(config, splits=("validation", "dev", "test", "train")):
    # Salesforce/ContextualBench hosts MuSiQue and 2WikiMultihopQA as parquet configs
    from datasets import load_dataset
    last = None
    for sp in splits:
        try:
            return load_dataset("Salesforce/ContextualBench", config, split=sp)
        except Exception as e:
            last = e
    raise RuntimeError(f"{config}: {repr(last)[:300]}")

def rows_of(ds, a, b):
    return [dict(r) for r in ds.select(range(a, min(b, len(ds))))]

In [6]:
# ---- 6. Retriever: dense search over the pooled paragraphs (numpy is enough at this scale) ----
def make_bge_embed_fn(model_name, device="cuda"):
    from sentence_transformers import SentenceTransformer
    m = SentenceTransformer(model_name, device=device)
    m.max_seq_length = 256
    prefix = "Represent this sentence for searching relevant passages: "
    def fn(texts, is_query):
        if is_query:
            texts = [prefix + t for t in texts]
        return m.encode(list(texts), batch_size=64 if is_query else 128, normalize_embeddings=True,
                        convert_to_numpy=True, show_progress_bar=(not is_query and len(texts) > 500))
    fn.model = m
    return fn

class Retriever:
    def __init__(self, examples, embed_fn, cache_dir=None, name=""):
        seen = {}
        self.docs = []
        for ex in examples:
            for title, text in ex["paragraphs"]:
                key = (title, text)
                if key not in seen:
                    seen[key] = len(self.docs)
                    self.docs.append({"title": title, "text": text})
        texts = [f"{d['title']}: {d['text']}" for d in self.docs]
        h = hashlib.md5("\n".join(texts).encode("utf-8")).hexdigest()[:12]
        cache = os.path.join(cache_dir, f"{name}_{h}.npy") if cache_dir else None
        if cache and os.path.exists(cache):
            E = np.load(cache)
        else:
            E = np.asarray(embed_fn(texts, False), dtype=np.float32)
            if cache:
                np.save(cache, E)
        self.E = E.astype(np.float32)
        self.embed_fn = embed_fn
        self.name = name

    def search(self, queries, k):
        Q = np.asarray(self.embed_fn(list(queries), True), dtype=np.float32)
        S = Q @ self.E.T
        k = min(k, S.shape[1])
        idx = np.argpartition(-S, k - 1, axis=1)[:, :k]
        out = []
        for r in range(S.shape[0]):
            order = idx[r][np.argsort(-S[r, idx[r]])]
            out.append([int(i) for i in order])
        return out

def fmt_passages(docs, max_words):
    lines = []
    for i, d in enumerate(docs, 1):
        words = d["text"].split()
        text = " ".join(words[:max_words]) + (" ..." if len(words) > max_words else "")
        lines.append(f"[{i}] {d['title']}: {text}")
    return "\n".join(lines)

In [7]:
# ---- 7. LLM engine: 4-bit Hugging Face model, batched greedy chat generation, optional LoRA adapter ----
class LLM:
    def __init__(self, model_id, adapter=None, device_map=None, batch_size=8):
        import torch
        from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
        self.torch = torch
        self.model_id, self.adapter, self.bs = model_id, adapter, batch_size
        bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                 bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
        self.tok = AutoTokenizer.from_pretrained(model_id)
        self.tok.padding_side = "left"          # required for batched generation
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token
        kw = {}
        if device_map == "auto":
            kw["max_memory"] = {0: "13GiB", 1: "14GiB"}
        t = time.time()
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id, quantization_config=bnb, device_map=(device_map or {"": 0}),
            torch_dtype=torch.float16, **kw)
        if adapter:
            from peft import PeftModel
            self.model = PeftModel.from_pretrained(self.model, adapter)
        self.model.eval()
        eos = {self.tok.eos_token_id, self.tok.convert_tokens_to_ids("<|im_end|>")}
        self.eos_ids = [i for i in eos if i is not None and i >= 0]
        used = sum(torch.cuda.memory_allocated(i) for i in range(torch.cuda.device_count())) / 2**30
        print(f"loaded {model_id}{' + ' + os.path.basename(adapter) if adapter else ''} in {time.time()-t:.0f}s, GPU mem {used:.1f} GiB")

    def _gen(self, prompts, max_new_tokens):
        torch = self.torch
        enc = self.tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(self.model.device)
        with torch.inference_mode():
            out = self.model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                                      temperature=None, top_p=None, top_k=None, repetition_penalty=1.0,
                                      pad_token_id=self.tok.pad_token_id, eos_token_id=self.eos_ids)
        gen = out[:, enc["input_ids"].shape[1]:]
        return self.tok.batch_decode(gen, skip_special_tokens=True)

    def chat(self, batch_messages, max_new_tokens=96, desc=""):
        if not batch_messages:
            return []
        from tqdm.auto import tqdm
        torch = self.torch
        prompts = [self.tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in batch_messages]
        order = list(np.argsort([len(p) for p in prompts])[::-1])   # longest first: OOM shows up immediately
        outs = [None] * len(prompts)
        i, bs = 0, self.bs
        pbar = tqdm(total=len(prompts), desc=desc, leave=False)
        while i < len(order):
            idx = order[i:i + bs]
            try:
                res = self._gen([prompts[j] for j in idx], max_new_tokens)
            except torch.cuda.OutOfMemoryError:
                gc.collect(); torch.cuda.empty_cache()
                if bs == 1:
                    raise
                bs = max(1, bs // 2); self.bs = bs
                print(f"  OOM -> batch size {bs}")
                continue
            for j, r in zip(idx, res):
                outs[j] = r
            i += len(idx); pbar.update(len(idx))
        pbar.close()
        return outs

    def close(self):
        torch = self.torch
        try:
            del self.model
        except Exception:
            pass
        gc.collect(); torch.cuda.empty_cache()

def open_llm(model_key, adapter=None):
    mid = CFG["MODELS"][model_key.split("-")[0]]
    dm = "auto" if model_key.startswith("14B") else None
    return LLM(mid, adapter=adapter, device_map=dm, batch_size=CFG["BATCH"][model_key.split("-")[0]])

def sanity_check(llm):
    out = llm.chat([[{"role": "user", "content": "What is the capital of France? Answer with one word."}]], 12)[0]
    print("sanity:", repr(out))
    if "paris" not in out.lower():
        print("WARNING: unexpected output. fp16 on T4 can overflow for some checkpoints; try another model size/family.")

In [8]:
# ---- 8. Prompts, tool-call parser, agent loop, baselines, scoring, failure taxonomy ----
SYSTEM_AGENT = "\n".join([
    "You are a research agent that answers multi-hop questions by searching a Wikipedia paragraph corpus.",
    "",
    "On every turn reply with exactly ONE JSON object and nothing else. Two tools exist:",
    '  {"tool": "search", "query": "<search query>"}   -> returns the top passages for the query',
    '  {"tool": "finish", "answer": "<final answer>"}   -> ends the task',
    "",
    "Rules:",
    "- Search for one fact at a time. Use entities found in earlier passages to write the next query.",
    "- Do not repeat a query you already used.",
    "- Finish as soon as the passages contain the evidence. Never guess before searching.",
    "- The final answer must be short: an entity name, a date, a number, or yes/no. No explanation.",
])
SYSTEM_DECOMPOSE = SYSTEM_AGENT + "\n" + "\n".join([
    "",
    "Strategy:",
    "- First work out which facts the question needs (for example: who is X, then what is Y of that person).",
    "- Query with specific names and keywords, not the whole question.",
    "- For comparison questions, look up each entity separately, then compare.",
])
PROMPTS = {"base": SYSTEM_AGENT, "decompose": SYSTEM_DECOMPOSE}
SYSTEM_PLAIN = "Answer the question with a short phrase only (an entity name, a date, a number, or yes/no). No explanation."
FINAL_NUDGE = "\n\n(Search budget used up: you must reply now with the finish tool.)"
INVALID_MSG = 'Invalid reply. Respond with exactly one JSON object, either {"tool": "search", "query": "..."} or {"tool": "finish", "answer": "..."}.'

_DEC = json.JSONDecoder()

def parse_action(text):
    # returns (action_or_None, clean). clean = the reply was nothing but the JSON object.
    if text is None:
        return None, False
    t = text.strip()
    t = re.sub(r"^```(?:json)?\s*", "", t)
    t = re.sub(r"\s*```$", "", t)
    i = t.find("{")
    if i < 0:
        return None, False
    try:
        obj, end = _DEC.raw_decode(t[i:])
    except Exception:
        return None, False
    if not isinstance(obj, dict):
        return None, False
    clean = (i == 0 and t[i + end:].strip() == "")
    tool = str(obj.get("tool", "")).lower().strip()
    if tool == "search":
        q = obj.get("query")
        if isinstance(q, str) and q.strip():
            return {"tool": "search", "query": q.strip()}, clean
    elif tool == "finish":
        a = obj.get("answer")
        if isinstance(a, (str, int, float)) and not isinstance(a, bool) and str(a).strip():
            return {"tool": "finish", "answer": str(a).strip()}, clean
    return None, False

def tag_failure(t):
    # one primary failure tag per trace (priority order). 'correct' if F1 >= 0.5.
    if t["f1"] >= 0.5:
        return "correct"
    if t["final"] is None:
        return "invalid_format" if t["n_invalid"] > 0 else "no_finish_loop"
    if t["n_invalid"] > 0:
        return "invalid_format"
    qs = [normalize_answer(q) for q in t["queries"]]
    if len(set(qs)) < len(qs):
        return "redundant_query"
    gr = t["gold_recall"]
    if t["n_search"] <= 1 and gr is not None and gr < 1.0:
        return "premature_answer"
    if gr is not None and gr < 1.0:
        return "retrieval_miss"
    return "reasoning_error"

def make_trace(ex, cond, mk, ds, final, messages, queries, retrieved, n_turns, n_invalid, n_unclean, forced):
    em, f1, acc = score_answer(final, ex["answers"])
    uniq = list(dict.fromkeys(retrieved))
    gold = ex["gold_titles"]
    gr = (sum(g in set(uniq) for g in gold) / len(gold)) if gold else None
    t = dict(id=ex["id"], dataset=ds, cond=cond, model=mk, question=ex["question"], answers=ex["answers"],
             final=final, em=em, f1=f1, acc=acc, gold_titles=gold, gold_recall=gr, queries=queries,
             n_search=len(queries), retrieved=uniq, n_turns=n_turns, n_invalid=n_invalid,
             n_unclean=n_unclean, forced=forced, messages=messages)
    t["tag"] = tag_failure(t)
    return t

def run_agent(llm, examples, retriever, mk="", ds="", top_k=3, max_steps=6, max_new_tokens=96,
              prompt="base", passage_words=90, cond="agentic"):
    system = PROMPTS[prompt]
    S = [dict(ex=ex, messages=[{"role": "system", "content": system},
                                {"role": "user", "content": "Question: " + ex["question"]}],
              done=False, final=None, queries=[], retrieved=[], n_invalid=0, n_unclean=0, n_turns=0, forced=False)
         for ex in examples]
    for step in range(max_steps):
        active = [s for s in S if not s["done"]]
        if not active:
            break
        last = (step == max_steps - 1)
        if last:
            for s in active:
                s["messages"][-1]["content"] += FINAL_NUDGE
                s["forced"] = True
        outs = llm.chat([s["messages"] for s in active], max_new_tokens=max_new_tokens,
                        desc=f"{cond} {mk} {ds} step {step+1}/{max_steps} ({len(active)} active)")
        pending = []
        for s, o in zip(active, outs):
            s["n_turns"] += 1
            act, clean = parse_action(o)
            if act is None:
                s["n_invalid"] += 1
                s["messages"].append({"role": "assistant", "content": (o or "").strip()})
                if last:
                    s["done"] = True
                else:
                    s["messages"].append({"role": "user", "content": INVALID_MSG})
                continue
            if not clean:
                s["n_unclean"] += 1
            s["messages"].append({"role": "assistant", "content": json.dumps(act, ensure_ascii=False)})
            if act["tool"] == "finish":
                s["final"], s["done"] = act["answer"], True
            else:
                s["queries"].append(act["query"])
                if last:
                    s["done"] = True
                else:
                    pending.append((s, act["query"]))
        if pending:
            hits = retriever.search([q for _, q in pending], top_k)
            for (s, _), h in zip(pending, hits):
                docs = [retriever.docs[i] for i in h]
                s["retrieved"].extend(d["title"] for d in docs)
                s["messages"].append({"role": "user", "content": "Observation:\n" + fmt_passages(docs, passage_words)})
    return [make_trace(s["ex"], cond, mk, ds, s["final"], s["messages"], s["queries"], s["retrieved"],
                       s["n_turns"], s["n_invalid"], s["n_unclean"], s["forced"]) for s in S]

def clean_plain_answer(text):
    a = (text or "").strip().split("\n")[0].strip()
    a = re.sub(r"^(answer\s*[:\-]\s*)", "", a, flags=re.I).strip().strip('"').strip("'")
    return a.rstrip(".").strip() or None

def run_no_retrieval(llm, examples, mk="", ds="", max_new_tokens=32, **_):
    msgs = [[{"role": "system", "content": SYSTEM_PLAIN}, {"role": "user", "content": "Question: " + ex["question"]}] for ex in examples]
    outs = llm.chat(msgs, max_new_tokens, desc=f"no_retrieval {mk} {ds}")
    return [make_trace(ex, "no_retrieval", mk, ds, clean_plain_answer(o), m + [{"role": "assistant", "content": o}], [], [], 1, 0, 0, False)
            for ex, m, o in zip(examples, msgs, outs)]

def run_fixed_rag(llm, examples, retriever, mk="", ds="", k=5, passage_words=90, max_new_tokens=32, **_):
    hits = retriever.search([ex["question"] for ex in examples], k)
    msgs, rets = [], []
    for ex, h in zip(examples, hits):
        docs = [retriever.docs[i] for i in h]
        rets.append([d["title"] for d in docs])
        msgs.append([{"role": "system", "content": SYSTEM_PLAIN},
                     {"role": "user", "content": "Passages:\n" + fmt_passages(docs, passage_words) + "\n\nQuestion: " + ex["question"]}])
    outs = llm.chat(msgs, max_new_tokens, desc=f"fixed_rag {mk} {ds}")
    return [make_trace(ex, "fixed_rag", mk, ds, clean_plain_answer(o), m + [{"role": "assistant", "content": o}],
                       [ex["question"]], r, 1, 0, 0, False)
            for ex, m, o, r in zip(examples, msgs, outs, rets)]

def summarize(traces):
    n = len(traces)
    if n == 0:
        return {}
    grs = [t["gold_recall"] for t in traces if t["gold_recall"] is not None]
    turns = sum(t["n_turns"] for t in traces)
    d = dict(n=n,
             EM=float(np.mean([t["em"] for t in traces])), F1=float(np.mean([t["f1"] for t in traces])),
             Acc=float(np.mean([t["acc"] for t in traces])),
             gold_recall=float(np.mean(grs)) if grs else float("nan"),
             avg_searches=float(np.mean([t["n_search"] for t in traces])),
             valid_call_rate=1.0 - sum(t["n_invalid"] for t in traces) / max(1, turns),
             clean_format_rate=1.0 - sum(t["n_invalid"] + t["n_unclean"] for t in traces) / max(1, turns),
             finish_rate=float(np.mean([t["final"] is not None for t in traces])))
    lo, hi = bootstrap_ci([t["f1"] for t in traces])
    d["F1_lo"], d["F1_hi"] = float(lo), float(hi)
    return d

In [9]:
# ---- 9. Runner helpers (resumable: a finished condition is never recomputed) ----
def trace_path(cond, mk, ds):
    return os.path.join(OUT, "traces", f"{cond}__{mk}__{ds}.jsonl")

def agent_kwargs():
    return dict(top_k=CFG["TOP_K"], max_steps=CFG["MAX_STEPS"], max_new_tokens=CFG["MAX_NEW_TOK"],
                prompt=CFG["PROMPT"], passage_words=CFG["PASSAGE_WORDS"])

def run_and_save(llm, cond, mk, ds, examples, retr):
    p = trace_path(cond, mk, ds)
    if os.path.exists(p) and not CFG["FORCE"]:
        print(f"[skip] {cond:13s} {mk:7s} {ds:11s} (already done)")
        return load_jsonl(p)
    check_budget()
    t0 = time.time()
    if cond == "no_retrieval":
        tr = run_no_retrieval(llm, examples, mk=mk, ds=ds)
    elif cond == "fixed_rag":
        tr = run_fixed_rag(llm, examples, retr, mk=mk, ds=ds, k=CFG["FIXED_RAG_K"], passage_words=CFG["PASSAGE_WORDS"])
    elif cond == "agentic":
        tr = run_agent(llm, examples, retr, mk=mk, ds=ds, **agent_kwargs())
    else:
        raise ValueError(cond)
    save_jsonl(p, tr)
    s = summarize(tr)
    print(f"[done] {cond:13s} {mk:7s} {ds:11s} n={s['n']:4d} EM={s['EM']:.3f} F1={s['F1']:.3f} "
          f"recall={s['gold_recall']:.3f} searches={s['avg_searches']:.2f} valid={s['valid_call_rate']:.3f} "
          f"({(time.time()-t0)/60:.1f} min)")
    return tr

def pending(conds_ds, mk):
    return [(c, d) for c, d in conds_ds if CFG["FORCE"] or not os.path.exists(trace_path(c, mk, d))]

## Stage 0 - Data and retrieval index
HotpotQA (distractor setting, `hotpotqa/hotpot_qa`) is the main dataset; MuSiQue and 2WikiMultihopQA (from `Salesforce/ContextualBench`) are used **only for testing generalisation**, never for training. Each dataset gets its own pooled paragraph corpus built from its sampled questions (see the limitation note at the top).

In [10]:
# ---- Stage 0: build question sets + retrieval indexes ----
STOPPED = False
EX, RETR = {}, {}
need_data = any(s in STAGES for s in ["tune", "base_eval", "traces", "finetune", "ft_eval", "ablate"])
TEACHER_ALL = os.path.join(OUT, "sft", "teacher_traces_all.jsonl")
SFT_ALL = os.path.join(OUT, "sft", "sft_all.jsonl")
SFT_FILTERED = os.path.join(OUT, "sft", "sft_filtered.jsonl")

if need_data:
    embed_fn = make_bge_embed_fn(CFG["EMB_MODEL"])
    hp_dev = load_hotpot_split("validation").shuffle(seed=SEED)
    EX["hotpot_dev"] = [conv_hotpot(r) for r in rows_of(hp_dev, 0, CFG["N_EVAL"])]
    if "tune" in STAGES:
        EX["hotpot_tune"] = [conv_hotpot(r) for r in rows_of(hp_dev, CFG["N_EVAL"], CFG["N_EVAL"] + CFG["N_TUNE"])]
    if ("traces" in STAGES or "ablate" in STAGES) and (not os.path.exists(SFT_FILTERED) or CFG["FORCE"]):
        hp_tr = load_hotpot_split("train").shuffle(seed=SEED)
        EX["hotpot_train"] = [conv_hotpot(r) for r in rows_of(hp_tr, 0, CFG["N_TRAIN_Q"])]
    if any(s in STAGES for s in ["base_eval", "ft_eval", "ablate"]):
        for name, cfgname, conv, only_answerable in [("musique", "MuSiQue", conv_musique, True),
                                                      ("twowiki", "2WikiMultihopQA", conv_2wiki, False)]:
            try:
                d = load_cb(cfgname)
                if only_answerable:
                    d = d.filter(lambda e: bool(e.get("answerable", True)))
                d = d.shuffle(seed=SEED)
                EX[name] = [conv(r) for r in rows_of(d, 0, CFG["N_OOD"])]
            except Exception as e:
                print(f"[warn] out-of-domain set {name} skipped: {repr(e)[:200]}")
    for name, exs in EX.items():
        RETR[name] = Retriever(exs, embed_fn, cache_dir=os.path.join(OUT, "emb"), name=name)
        print(f"{name:12s} questions={len(exs):5d}  corpus paragraphs={len(RETR[name].docs):6d}")
    # quick retrieval sanity: how often does the plain question retrieve both gold paragraphs in the top 5?
    chk = EX["hotpot_dev"][:100]
    hits = RETR["hotpot_dev"].search([e["question"] for e in chk], 5)
    rec = np.mean([np.mean([g in {RETR["hotpot_dev"].docs[i]["title"] for i in h} for g in e["gold_titles"]]) for e, h in zip(chk, hits)])
    print(f"retrieval sanity: gold-paragraph recall@5 of the raw question = {rec:.2f}")

EVAL_DS = ["hotpot_dev"] + [n for n in ("musique", "twowiki") if n in EX]
COND3 = ["no_retrieval", "fixed_rag", "agentic"]
tuned_path = os.path.join(OUT, "tuned.json")
if os.path.exists(tuned_path) and "tune" not in STAGES:
    CFG.update(json.load(open(tuned_path)))
    print("loaded tuned settings:", json.load(open(tuned_path)))
print("evaluation sets:", EVAL_DS)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

2WikiMultihopQA/test-00000-of-00001.parq(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/167454 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/12576 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/12576 [00:00<?, ? examples/s]

hotpot_dev   questions=   24  corpus paragraphs=   240
hotpot_train questions=   60  corpus paragraphs=   598
musique      questions=   12  corpus paragraphs=   240
twowiki      questions=   12  corpus paragraphs=   102
retrieval sanity: gold-paragraph recall@5 of the raw question = 0.88
evaluation sets: ['hotpot_dev', 'musique', 'twowiki']


## Stage 1 (optional) - tune prompt and top-k on held-out questions
Uses `N_TUNE` dev questions that are **not** part of the final evaluation. This is the cheapest way to get a better agent without touching the test set. Add `"tune"` to `STAGES` to enable it; the winner is stored in `tuned.json` and reused automatically.

In [11]:
# ---- Stage 1: sweep prompt variant x top-k on the held-out tuning questions ----
if "tune" in STAGES and not STOPPED:
    try:
        llm = open_llm(CFG["TUNE_MODEL"]); sanity_check(llm)
        rows = []
        for variant in ["base", "decompose"]:
            for k in [2, 3, 5]:
                check_budget()
                kw = dict(agent_kwargs(), prompt=variant, top_k=k)
                tr = run_agent(llm, EX["hotpot_tune"], RETR["hotpot_tune"], mk=CFG["TUNE_MODEL"], ds="hotpot_tune", **kw)
                s = summarize(tr)
                rows.append(dict(prompt=variant, top_k=k, F1=s["F1"], EM=s["EM"], recall=s["gold_recall"], valid=s["valid_call_rate"]))
                print(rows[-1])
        llm.close()
        best = max(rows, key=lambda r: r["F1"])
        CFG["PROMPT"], CFG["TOP_K"] = best["prompt"], best["top_k"]
        json.dump({"PROMPT": best["prompt"], "TOP_K": best["top_k"]}, open(tuned_path, "w"))
        json.dump(rows, open(os.path.join(OUT, "logs", "tune_sweep.json"), "w"), indent=1)
        print("BEST:", best)
    except BudgetExceeded as e:
        print(e); STOPPED = True

## Stage 2 - Baselines, small-model benchmark, and teacher traces
For each model size (loaded **once**) the notebook runs `no_retrieval`, `fixed_rag` and `agentic` on HotpotQA dev and on the out-of-domain sets. The teacher model additionally solves `N_TRAIN_Q` HotpotQA *train* questions; those traces become the fine-tuning data.

In [12]:
# ---- Stage 2: base models -> evaluation + teacher trace generation ----
def gen_teacher_traces(llm, mk, chunk=200):
    exs, retr = EX["hotpot_train"], RETR["hotpot_train"]
    parts = []
    for ci, a in enumerate(range(0, len(exs), chunk)):
        part = os.path.join(OUT, "sft", f"teacher_{mk}_part{ci:03d}.jsonl")
        parts.append(part)
        if os.path.exists(part) and not CFG["FORCE"]:
            continue
        check_budget()
        t0 = time.time()
        tr = run_agent(llm, exs[a:a + chunk], retr, mk=mk, ds="hotpot_train", **agent_kwargs())
        save_jsonl(part, tr)
        print(f"  teacher chunk {ci}: {len(tr)} traces, F1={np.mean([t['f1'] for t in tr]):.3f} ({(time.time()-t0)/60:.1f} min)")
    allt = [t for p in parts for t in load_jsonl(p)]
    save_jsonl(TEACHER_ALL, allt)
    return allt

def good_trace(t):
    # keep only clean, correct trajectories (optionally also: all gold paragraphs were actually retrieved)
    ok = (t["final"] is not None and t["n_invalid"] == 0 and not t["forced"] and t["n_search"] >= 1 and t["f1"] >= 0.8)
    if CFG["REQUIRE_GOLD_RECALL"]:
        ok = ok and (t["gold_recall"] is None or t["gold_recall"] >= 1.0)
    return ok

def build_sft_files():
    allt = load_jsonl(TEACHER_ALL)
    seen, filt = set(), []
    for t in allt:
        if t["id"] in seen:
            continue
        seen.add(t["id"])
        if good_trace(t):
            filt.append({"id": t["id"], "messages": t["messages"]})
    rng = random.Random(SEED)
    rng.shuffle(filt)
    filt = filt[:CFG["MAX_TRAIN_TRACES"]]
    unf = [{"id": t["id"], "messages": [m for m in t["messages"]]} for t in allt if t["n_invalid"] == 0 and t["final"] is not None]
    rng.shuffle(unf)
    unf = unf[:max(1, len(filt))]
    save_jsonl(SFT_FILTERED, filt); save_jsonl(SFT_ALL, unf)
    if not filt:
        print("[warn] 0 traces survived the filter. Check retrieval sanity above, or set CFG['REQUIRE_GOLD_RECALL']=False.")
    elif len(filt) < 300:
        print(f"[warn] only {len(filt)} filtered traces. Consider a larger N_TRAIN_Q or CFG['REQUIRE_GOLD_RECALL']=False.")
    tags = Counter(t["tag"] for t in allt)
    print(f"teacher traces: {len(allt)} | kept (filtered): {len(filt)} ({len(filt)/max(1,len(allt)):.0%}) | unfiltered control set: {len(unf)}")
    print("teacher outcome tags:", dict(tags))
    return filt

if any(s in STAGES for s in ["base_eval", "traces"]) and not STOPPED:
    for mk in CFG["EVAL_MODELS"]:
        todo = pending([(c, d) for d in EVAL_DS for c in COND3], mk) if "base_eval" in STAGES else []
        need_teacher = ("traces" in STAGES and mk == CFG["TEACHER"] and (not os.path.exists(SFT_FILTERED) or CFG["FORCE"]))
        if not todo and not need_teacher:
            print(f"[skip] {mk}: everything already computed")
            continue
        llm = None
        try:
            llm = open_llm(mk); sanity_check(llm)
            if "base_eval" in STAGES:
                for ds in EVAL_DS:
                    for c in COND3:
                        run_and_save(llm, c, mk, ds, EX[ds], RETR[ds])
            if need_teacher:
                gen_teacher_traces(llm, mk)
                build_sft_files()
        except BudgetExceeded as e:
            print(e); STOPPED = True
        finally:
            if llm is not None:
                llm.close()
        if STOPPED:
            break

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

agentic 3B hotpot_train step 3/6 (31 active):   0%|          | 0/31 [00:00<?, ?it/s]

agentic 3B hotpot_train step 4/6 (7 active):   0%|          | 0/7 [00:00<?, ?it/s]

agentic 3B hotpot_train step 5/6 (2 active):   0%|          | 0/2 [00:00<?, ?it/s]

agentic 3B hotpot_train step 6/6 (2 active):   0%|          | 0/2 [00:00<?, ?it/s]

  teacher chunk 0: 60 traces, F1=0.595 (1.1 min)
[warn] only 26 filtered traces. Consider a larger N_TRAIN_Q or CFG['REQUIRE_GOLD_RECALL']=False.
teacher traces: 60 | kept (filtered): 26 (43%) | unfiltered control set: 26
teacher outcome tags: {'correct': 38, 'reasoning_error': 11, 'premature_answer': 4, 'retrieval_miss': 4, 'invalid_format': 1, 'redundant_query': 2}


## Stage 3 - QLoRA on filtered teacher traces
Loss is computed **only on the assistant turns** (the JSON tool calls). Observations, system prompt and questions are context. A custom training loop is used instead of `Trainer` so that Kaggle's two visible GPUs do not trigger `DataParallel` problems with 4-bit models.

In [13]:
# ---- SFT data construction (assistant-only loss) and QLoRA training loop ----
def build_sft_example(tok, messages, max_len):
    full = tok.apply_chat_template(messages, tokenize=False)
    ids = tok(full, add_special_tokens=False)["input_ids"]
    if len(ids) > max_len:
        return None
    labels = [-100] * len(ids)
    for i, m in enumerate(messages):
        if m["role"] != "assistant":
            continue
        pre = tok.apply_chat_template(messages[:i], tokenize=False, add_generation_prompt=True)
        upto = tok.apply_chat_template(messages[:i + 1], tokenize=False)
        a = len(tok(pre, add_special_tokens=False)["input_ids"])
        b = len(tok(upto, add_special_tokens=False)["input_ids"])
        labels[a:b] = ids[a:b]
    if all(l == -100 for l in labels):
        return None
    return ids, labels

def train_lora(base_key, rows, out_dir, tag):
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, get_cosine_schedule_with_warmup
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    from tqdm.auto import tqdm
    mid = CFG["MODELS"][base_key]
    tok = AutoTokenizer.from_pretrained(mid)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    data = [build_sft_example(tok, r["messages"], CFG["MAX_SEQ_LEN"]) for r in rows]
    n_drop = sum(d is None for d in data)
    data = [d for d in data if d is not None]
    assert data, "no training examples left after length filtering"
    toks = sum(len(d[0]) for d in data); trained = sum(sum(l != -100 for l in d[1]) for d in data)
    print(f"[{tag}] {len(data)} examples ({n_drop} dropped for length), {toks/1e6:.2f}M tokens, {trained/1e3:.0f}k supervised tokens")

    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    model = AutoModelForCausalLM.from_pretrained(mid, quantization_config=bnb, device_map={"": 0}, torch_dtype=torch.float16)
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True,
                                            gradient_checkpointing_kwargs={"use_reentrant": False})
    lcfg = LoraConfig(r=CFG["LORA_R"], lora_alpha=CFG["LORA_ALPHA"], lora_dropout=CFG["LORA_DROPOUT"], bias="none",
                      task_type="CAUSAL_LM",
                      target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
    model = get_peft_model(model, lcfg)
    model.print_trainable_parameters()

    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=CFG["LR"], weight_decay=0.0)
    ga = CFG["GRAD_ACC"]
    total_steps = max(1, math.ceil(len(data) * CFG["EPOCHS"] / ga))
    sched = get_cosine_schedule_with_warmup(opt, max(1, int(0.05 * total_steps)), total_steps)
    scaler = torch.amp.GradScaler("cuda")
    model.train()
    log, step, acc_loss, acc_n, skipped = [], 0, 0.0, 0, 0
    t0 = time.time()
    for ep in range(CFG["EPOCHS"]):
        order = list(range(len(data)))
        random.Random(SEED + ep).shuffle(order)
        for i, di in enumerate(tqdm(order, desc=f"{tag} epoch {ep+1}/{CFG['EPOCHS']}", leave=False)):
            ids, labels = data[di]
            x = torch.tensor([ids], device="cuda"); y = torch.tensor([labels], device="cuda")
            with torch.autocast("cuda", dtype=torch.float16):
                loss = model(input_ids=x, attention_mask=torch.ones_like(x), labels=y).loss
            if not torch.isfinite(loss):
                skipped += 1
                continue
            scaler.scale(loss / ga).backward()
            acc_loss += float(loss); acc_n += 1
            last_in_epoch = (i == len(order) - 1)
            if acc_n % ga == 0 or last_in_epoch:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
                step += 1
                log.append(dict(step=step, epoch=ep + 1, loss=acc_loss / max(1, acc_n), lr=sched.get_last_lr()[0]))
                acc_loss, acc_n = 0.0, 0
                if step % 5 == 0:
                    print(f"  [{tag}] step {step}/{total_steps} loss={log[-1]['loss']:.4f} ({(time.time()-t0)/60:.1f} min)")
        check_budget(0.1)
    os.makedirs(out_dir, exist_ok=True)
    model.save_pretrained(out_dir); tok.save_pretrained(out_dir)
    json.dump(log, open(os.path.join(OUT, "logs", f"train_{tag}.json"), "w"))
    print(f"[{tag}] saved adapter -> {out_dir} | skipped non-finite batches: {skipped} | {(time.time()-t0)/60:.1f} min")
    del model, opt
    gc.collect(); torch.cuda.empty_cache()
    return log

In [14]:
# ---- Stage 3: train the main adapter ----
def adapter_dir(name):
    return os.path.join(OUT, "adapters", name)

def adapter_ready(name):
    return os.path.exists(os.path.join(adapter_dir(name), "adapter_config.json"))

FT_NAME = f"{CFG['FT_BASE']}-ft"
if "finetune" in STAGES and not STOPPED:
    if adapter_ready(FT_NAME) and not CFG["FORCE"]:
        print(f"[skip] adapter {FT_NAME} already trained")
    elif not os.path.exists(SFT_FILTERED):
        print("[warn] no SFT file yet - run the 'traces' stage first.")
    else:
        try:
            train_lora(CFG["FT_BASE"], load_jsonl(SFT_FILTERED), adapter_dir(FT_NAME), FT_NAME)
        except BudgetExceeded as e:
            print(e); STOPPED = True

[3B-ft] 26 examples (0 dropped for length), 0.02M tokens, 1k supervised tokens


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


3B-ft epoch 1/1:   0%|          | 0/26 [00:00<?, ?it/s]

/tmp/ipykernel_48/1708856285.py:69: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  acc_loss += float(loss); acc_n += 1


  [3B-ft] step 5/7 loss=0.2272 (0.7 min)
[3B-ft] saved adapter -> /kaggle/working/outputs/smoke/adapters/3B-ft | skipped non-finite batches: 0 | 0.8 min


## Stage 4 - Evaluate the fine-tuned student
Same questions, same retrieval, same prompt as the base model. The comparison base vs `-ft` on the **out-of-domain** sets is the result that shows whether the model learned the agent loop rather than memorising HotpotQA.

In [15]:
# ---- Stage 4: evaluate the fine-tuned adapter ----
def eval_adapter(name, datasets):
    todo = pending([("agentic", d) for d in datasets], name)
    if not todo:
        print(f"[skip] {name}: already evaluated")
        return
    llm = None
    try:
        llm = open_llm(CFG["FT_BASE"], adapter=adapter_dir(name)); sanity_check(llm)
        for ds in datasets:
            run_and_save(llm, "agentic", name, ds, EX[ds], RETR[ds])
    finally:
        if llm is not None:
            llm.close()

if "ft_eval" in STAGES and not STOPPED:
    if adapter_ready(FT_NAME):
        try:
            eval_adapter(FT_NAME, EVAL_DS)
        except BudgetExceeded as e:
            print(e); STOPPED = True
    else:
        print("[warn] adapter missing - run the 'finetune' stage first.")

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

loaded Qwen/Qwen2.5-3B-Instruct + 3B-ft in 5s, GPU mem 3.5 GiB


  0%|          | 0/1 [00:00<?, ?it/s]

sanity: 'Paris'


agentic 3B-ft hotpot_dev step 1/6 (24 active):   0%|          | 0/24 [00:00<?, ?it/s]

...

[done] agentic       3B-ft   musique     n=  12 EM=0.083 F1=0.234 recall=0.583 searches=2.42 valid=1.000 (0.6 min)


agentic 3B-ft twowiki step 1/6 (12 active):   0%|          | 0/12 [00:00<?, ?it/s]

agentic 3B-ft twowiki step 2/6 (12 active):   0%|          | 0/12 [00:00<?, ?it/s]

agentic 3B-ft twowiki step 3/6 (12 active):   0%|          | 0/12 [00:00<?, ?it/s]

[done] agentic       3B-ft   twowiki     n=  12 EM=0.250 F1=0.367 recall=0.771 searches=2.00 valid=1.000 (0.4 min)


## Stage 5 (optional) - ablations
Add `"ablate"` to `STAGES`. Trains smaller-data adapters (`ABL_SIZES`) and one adapter on **unfiltered** traces of the same size as the main set. Expected story for the write-up: more filtered data helps; training on unfiltered (including wrong) traces helps less or hurts.

In [16]:
# ---- Stage 5: ablations (training-set size, filtered vs unfiltered) ----
if "ablate" in STAGES and not STOPPED and os.path.exists(SFT_FILTERED):
    try:
        filt = load_jsonl(SFT_FILTERED)
        jobs = [(f"{CFG['FT_BASE']}-ft-n{n}", filt[:n]) for n in CFG["ABL_SIZES"] if n < len(filt)]
        if os.path.exists(SFT_ALL):
            jobs.append((f"{CFG['FT_BASE']}-ft-unfiltered", load_jsonl(SFT_ALL)))
        for name, rows in jobs:
            if not adapter_ready(name) or CFG["FORCE"]:
                train_lora(CFG["FT_BASE"], rows, adapter_dir(name), name)
            eval_adapter(name, EVAL_DS)
    except BudgetExceeded as e:
        print(e); STOPPED = True

## Stage 6 - Report: tables, figures, significance tests, `RESULTS.md`
Runs on whatever trace files exist, so you can also execute only this cell (`STAGES = ["report"]`) in a new session after attaching the earlier output as input.

In [17]:
# ---- Stage 6: report ----
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def load_all_traces():
    res = {}
    for p in sorted(glob.glob(os.path.join(OUT, "traces", "*.jsonl"))):
        cond, mk, ds = os.path.basename(p)[:-len(".jsonl")].split("__")
        res[(cond, mk, ds)] = load_jsonl(p)
    return res

def md_table(df, fmt="{:.3f}"):
    cols = list(df.columns)
    lines = ["| " + " | ".join(cols) + " |", "|" + "|".join(["---"] * len(cols)) + "|"]
    for _, r in df.iterrows():
        lines.append("| " + " | ".join(fmt.format(v) if isinstance(v, float) else str(v) for v in r.values) + " |")
    return "\n".join(lines)

def model_order(m):
    base = m.split("-")[0]
    return ({"3B": 0, "7B": 1, "14B": 2}.get(base, 9), m)

if "report" in STAGES:
    T = load_all_traces()
    if not T:
        print("No traces found in", os.path.join(OUT, "traces"))
    else:
        rows = []
        for (cond, mk, ds), tr in T.items():
            s = summarize(tr)
            rows.append(dict(dataset=ds, model=mk, condition=cond, n=s["n"], EM=s["EM"], F1=s["F1"], Acc=s["Acc"],
                             gold_recall=s["gold_recall"], searches=s["avg_searches"], valid_call=s["valid_call_rate"],
                             clean_fmt=s["clean_format_rate"], finish=s["finish_rate"], F1_lo=s["F1_lo"], F1_hi=s["F1_hi"]))
        df = pd.DataFrame(rows)
        df["_o"] = df["model"].map(lambda m: model_order(m))
        cond_rank = {"no_retrieval": 0, "fixed_rag": 1, "agentic": 2}
        df["_c"] = df["condition"].map(cond_rank)
        df = df.sort_values(["dataset", "_o", "_c"]).drop(columns=["_o", "_c"]).reset_index(drop=True)
        df.to_csv(os.path.join(OUT, "results_summary.csv"), index=False)
        main_md = md_table(df)
        print(main_md)

        # failure taxonomy (agentic runs only)
        TAGS = ["correct", "invalid_format", "no_finish_loop", "redundant_query", "premature_answer", "retrieval_miss", "reasoning_error"]
        frows = []
        for (cond, mk, ds), tr in T.items():
            if cond != "agentic":
                continue
            c = Counter(t["tag"] for t in tr)
            frows.append(dict(dataset=ds, model=mk, **{k: c.get(k, 0) / len(tr) for k in TAGS}))
        fdf = pd.DataFrame(frows)
        if len(fdf):
            fdf["_o"] = fdf["model"].map(lambda m: model_order(m))
            fdf = fdf.sort_values(["dataset", "_o"]).drop(columns="_o").reset_index(drop=True)
            fdf.to_csv(os.path.join(OUT, "failure_taxonomy.csv"), index=False)
            fail_md = md_table(fdf, "{:.2f}")
            print("\nFailure taxonomy (share of questions):\n" + fail_md)
        else:
            fail_md = "(no agentic runs yet)"

        # paired significance: base vs fine-tuned (F1 per question)
        sig_lines = []
        for ds in sorted({k[2] for k in T}):
            a, b = T.get(("agentic", CFG["FT_BASE"], ds)), T.get(("agentic", f"{CFG['FT_BASE']}-ft", ds))
            if a and b:
                bm = {t["id"]: t["f1"] for t in b}
                pairs = [(t["f1"], bm[t["id"]]) for t in a if t["id"] in bm]
                if pairs:
                    d, ci, p = paired_bootstrap([x for x, _ in pairs], [y for _, y in pairs])
                    sig_lines.append(f"- {ds}: F1 {CFG['FT_BASE']} -> {CFG['FT_BASE']}-ft = {d:+.3f}  (95% CI {ci[0]:+.3f} .. {ci[1]:+.3f}, P(diff<=0)={p:.3f}, n={len(pairs)})")
        sig_md = "\n".join(sig_lines) or "(need both base and fine-tuned agentic runs)"
        print("\nPaired bootstrap, base vs fine-tuned:\n" + sig_md)

        # ---- figures ----
        figdir = os.path.join(OUT, "figs")
        d0 = df[df["dataset"] == "hotpot_dev"]
        if len(d0):
            models = sorted(d0["model"].unique(), key=model_order)
            fig, ax = plt.subplots(figsize=(8, 4))
            w = 0.8 / 3
            for j, c in enumerate(["no_retrieval", "fixed_rag", "agentic"]):
                vals = [float(d0[(d0.model == m) & (d0.condition == c)]["F1"].mean()) if len(d0[(d0.model == m) & (d0.condition == c)]) else 0 for m in models]
                ax.bar(np.arange(len(models)) + j * w, vals, w, label=c)
            ax.set_xticks(np.arange(len(models)) + w); ax.set_xticklabels(models)
            ax.set_ylabel("F1"); ax.set_title("HotpotQA dev: F1 by condition and model"); ax.legend()
            fig.tight_layout(); fig.savefig(os.path.join(figdir, "fig1_conditions.png"), dpi=150); plt.close(fig)
        if len(fdf):
            f0 = fdf[fdf["dataset"] == "hotpot_dev"]
            if len(f0):
                fig, ax = plt.subplots(figsize=(8, 4))
                bottom = np.zeros(len(f0))
                for tg in TAGS[1:]:
                    ax.bar(f0["model"], f0[tg].values, bottom=bottom, label=tg); bottom += f0[tg].values
                ax.set_ylabel("share of questions"); ax.set_title("Agent failure taxonomy (HotpotQA dev)"); ax.legend(fontsize=7)
                fig.tight_layout(); fig.savefig(os.path.join(figdir, "fig2_failures.png"), dpi=150); plt.close(fig)
        a0 = df[df["condition"] == "agentic"]
        if len(a0):
            dss = [d for d in ["hotpot_dev", "musique", "twowiki"] if d in set(a0["dataset"])]
            models = sorted(a0["model"].unique(), key=model_order)
            fig, ax = plt.subplots(figsize=(8, 4))
            w = 0.8 / max(1, len(models))
            for j, m in enumerate(models):
                vals = [float(a0[(a0.model == m) & (a0.dataset == d)]["F1"].mean()) if len(a0[(a0.model == m) & (a0.dataset == d)]) else 0 for d in dss]
                ax.bar(np.arange(len(dss)) + j * w, vals, w, label=m)
            ax.set_xticks(np.arange(len(dss)) + 0.4 - w / 2); ax.set_xticklabels(dss)
            ax.set_ylabel("F1"); ax.set_title("Agentic F1: in-domain vs out-of-domain"); ax.legend(fontsize=8)
            fig.tight_layout(); fig.savefig(os.path.join(figdir, "fig3_generalisation.png"), dpi=150); plt.close(fig)
        lp = os.path.join(OUT, "logs", f"train_{CFG['FT_BASE']}-ft.json")
        if os.path.exists(lp):
            lg = json.load(open(lp))
            fig, ax = plt.subplots(figsize=(6, 3.5))
            ax.plot([r["step"] for r in lg], [r["loss"] for r in lg]); ax.set_xlabel("optimizer step"); ax.set_ylabel("loss")
            ax.set_title("QLoRA training loss"); fig.tight_layout(); fig.savefig(os.path.join(figdir, "fig4_loss.png"), dpi=150); plt.close(fig)

        # ---- RESULTS.md ----
        with open(os.path.join(OUT, "RESULTS.md"), "w") as f:
            f.write(f"# Results ({RUN_NAME})\n\nSettings: TOP_K={CFG['TOP_K']}, PROMPT={CFG['PROMPT']}, MAX_STEPS={CFG['MAX_STEPS']}, "
                    f"N_EVAL={CFG['N_EVAL']}, N_OOD={CFG['N_OOD']}, N_TRAIN_Q={CFG['N_TRAIN_Q']}\n\n")
            f.write("## Main table\n\n" + main_md + "\n\n## Failure taxonomy (share of questions)\n\n" + fail_md +
                    "\n\n## Base vs fine-tuned (paired bootstrap on per-question F1)\n\n" + sig_md + "\n")
        print("\nSaved RESULTS.md, results_summary.csv, failure_taxonomy.csv and figures in", OUT)

        # ---- look at real traces: this is where the insights come from ----
        def show_trace(t, maxc=350):
            print(f"\n=== [{t['model']} | {t['dataset']} | tag={t['tag']}] F1={t['f1']:.2f}")
            print("Q:", t["question"], "| gold:", t["answers"][0], "| final:", t["final"])
            for m in t["messages"][2:]:
                print(f"  {m['role'][:4]}: {m['content'][:maxc].replace(chr(10), ' / ')}")
        key = ("agentic", f"{CFG['FT_BASE']}-ft", "hotpot_dev")
        key = key if key in T else ("agentic", CFG["FT_BASE"], "hotpot_dev")
        if key in T:
            fails = [t for t in T[key] if t["tag"] != "correct"][:2]
            wins = [t for t in T[key] if t["tag"] == "correct" and t["n_search"] >= 2][:1]
            for t in wins + fails:
                show_trace(t)
        shutil.make_archive(os.path.join(BASE_OUT, f"outputs_{RUN_NAME}_bundle"), "zip", OUT)
        print("\nZip bundle:", os.path.join(BASE_OUT, f"outputs_{RUN_NAME}_bundle.zip"))

| dataset | model | condition | n | EM | F1 | Acc | gold_recall | searches | valid_call | clean_fmt | finish | F1_lo | F1_hi |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| hotpot_dev | 3B | no_retrieval | 24 | 0.208 | 0.304 | 0.208 | 0.000 | 0.000 | 1.000 | 1.000 | 1.000 | 0.146 | 0.464 |
| hotpot_dev | 3B | fixed_rag | 24 | 0.417 | 0.561 | 0.542 | 0.875 | 1.000 | 1.000 | 1.000 | 1.000 | 0.378 | 0.733 |
| hotpot_dev | 3B | agentic | 24 | 0.583 | 0.644 | 0.625 | 0.812 | 1.583 | 1.000 | 0.984 | 1.000 | 0.458 | 0.819 |
| hotpot_dev | 3B-ft | agentic | 24 | 0.625 | 0.686 | 0.667 | 0.875 | 1.792 | 1.000 | 0.970 | 1.000 | 0.497 | 0.853 |
| musique | 3B | no_retrieval | 12 | 0.000 | 0.201 | 0.000 | 0.000 | 0.000 | 1.000 | 1.000 | 1.000 | 0.042 | 0.375 |
| musique | 3B | fixed_rag | 12 | 0.250 | 0.430 | 0.250 | 0.681 | 1.000 | 1.000 | 1.000 | 1.000 | 0.200 | 0.667 |
| musique | 3B | agentic | 12 | 0.000 | 0.190 | 0.083 | 0.562 | 2.667 | 1.000 | 1.000 | 1.000 | 0.048 | 0.349 |
| 

## After the run: how to get the best results and write them up

* **Read in this order:** `RESULTS.md` -> `figs/` -> the printed example traces -> `traces/*.jsonl` (filter by `tag` to study one failure type).
* **Main claims that need no extra experiments:** (i) agentic vs fixed RAG vs no retrieval per model size; (ii) failure taxonomy shifts with model size; (iii) base vs fine-tuned on out-of-domain sets with a paired bootstrap CI.
* **If fine-tuning does not help much:** check `valid_call` / `clean_fmt` first (format gains show up there before accuracy), raise `N_TRAIN_Q` (more teacher traces), use `EPOCHS=3`, or lower `LR` to `1e-4` if loss is noisy.
* **If the agent is the weak link (low recall):** run the `tune` stage, try `TOP_K=5`, `MAX_STEPS=8`, and the `decompose` prompt.
* **Always report** n, the CI, the exact model ids, and the corpus limitation. Never tune on `hotpot_dev`; the tune stage uses separate questions for that reason.
* **Publish:** upload `adapters/3B-ft` to the Hugging Face Hub and the repo + `RESULTS.md` to GitHub.